# Config

In [204]:
!pip install nltk==3.9.1
!pip install mlflow==3.3.1

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [205]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [206]:
import pandas as pd
import os
from preprocess.preprocess import clean_text, check_deleted_expressions
from preprocess.translate import translator, gen_text_for_embedding, final_clean, detect_language
import time
import json

# 1) Preprocesamiento de los datos


In [6]:
# 1) Cargar datos
path = "/tmp/data"
path_analytics = "/tmp/analytics"
filePATH = os.path.join(path, "data_concatenada.xlsx")
df = pd.read_excel(filePATH,
                   usecols=["Código VRID", "Título", "Resumen", "Keywords", "Interdisciplinario", "Transdisciplinario", "Facultad del Proyecto",
                            "Depto Persona"]) \
       .fillna("")

# 2) Guardar qué secuencias de palabras del resumen serán eliminadas al aplicar get_expressions_to_delete()
list_texts = df["Resumen"].to_list()
df_deleted = check_deleted_expressions(list_texts)
savepath=os.path.join(path_analytics, "deleted_re.xlsx")
df_deleted.to_excel(savepath, index=False)

# 3) Preprocesar los datos
#Columnas que se van a preprocesar
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Título": "Titulo_trad",
    "Resumen": "Resumen_trad",
    "Keywords": "keywords_trad",
    "Facultad del Proyecto": "Facultad_del_Proyecto_trad",
    "Depto Persona": "Depto_Persona_trad",
}
#Preprocesamiento de datos
df[list(cols.values())] = df[list(cols.keys())].applymap(clean_text)
savepath=os.path.join(path, "data_clean.xlsx")
df.to_excel(savepath, index=False)

# 2) Traducción del texto

In [ ]:
#Crear columna de registro de idioma: 
# True: Texto en español, False: Texto en inglés
df["Español"]=detect_language(df["Resumen_trad"])

In [9]:
from transformers import MarianMTModel, MarianTokenizer

#1. Cargar modelo de traducción
model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
trans = translator(model, tokenizer)

#Columnas que se van a traducir
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Titulo_trad": "Titulo_trad",
    "Resumen_trad": "Resumen_trad",
    "keywords_trad": "keywords_trad",
    "Facultad_del_Proyecto_trad": "Facultad_del_Proyecto_trad",
    "Depto_Persona_trad": "Depto_Persona_trad",
}

#2. Traducción de columnas
#####Estoy trabajando en mejorar esta parte para que sea más rápida con paralelización por batches
start = time.time()
for src, dst in cols.items():
    df[dst] = trans.translate_parallel(df[src].to_list(), batch_size=8)
end = time.time()


#3.Guardado de resultados
savepath=os.path.join(path, "data_translated.xlsx")
df.to_excel(savepath, index=False)

print(f"Tiempo total de traducción: {end - start:.2f} segundos")

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Usando dispositivo: cuda


model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

Traduciendo: 100%|██████████| 119/119 [00:25<00:00,  4.64batch/s]


Tiempo total de traducción: 755.99 segundos


In [10]:
#3. Selección de columnas que se utilizarán en clasificador y concatenación
# Última limpieza antes de generar concatenación
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad", "Facultad_del_Proyecto_trad", "Depto_Persona_trad"]
for col in cols:
    df[col] = df[col].apply(final_clean)

#  Selección de columnas para embedding.
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]
element_names = ["title", "keywords", "abstract"]
df = gen_text_for_embedding(df, cols, element_names)

# Guardado de resultados
savepath=os.path.join(path, "data_translated_concat.xlsx")
df.to_excel(savepath, index=False)
savepath=os.path.join(path, "data_translated_concat.csv")
df.to_csv(savepath, index=False, encoding="utf-8-sig")

df.head()

,Código VRID,Interdisciplinario,Transdisciplinario,Título,Keywords,Resumen,Facultad del Proyecto,Depto Persona,Titulo_trad,Resumen_trad,keywords_trad,Facultad_del_Proyecto_trad,Depto_Persona_trad,Español,text_for_embedding_translated
0,217.173.049-1.0,SI,,PATRONES DE CRIANZA Y SOCIALIZACIÓN DE GÉNERO ...,,OBJETIVOS GENERALES: _x000D_\nDESCRIBIR LOS PR...,FACULTAD DE CIENCIAS SOCIALES,"DEPARTAMENTO DE ECONOMÍA, SIN INFORMACIÓN, ESC...",patterns of gender upbringing and socializatio...,general objectives: to describe the processes ...,,faculty of social sciences,"department of economics, without information, ...",True,title: patterns of gender upbringing and socia...
1,218.201.002-1.0,SI,,ADAPTACIÓN CULTURAL Y VALIDACIÓN DE LA ESCALA ...,"ESTILO DE VIDA, ADOLESCENTES _x000D_\n",PARA EVALUAR LOS COMPORTAMIENTOS RELACIONADOS ...,FACULTAD DE ENFERMERÍA,"DEPARTAMENTO DE CIENCIA ANIMAL, DEPARTAMENTO D...",cultural adaptation and validation of the life...,in order to evaluate the behaviors related to ...,"lifestyle, teens",faculty of nursing,"department of animal science, department of pl...",True,title: cultural adaptation and validation of t...
2,218.102.031-1.0IN,NO,,PROMOVIENDO LA REFLEXIÓN EN ESTUDIANTES DE PRE...,,EL PRESENTE PROYECTO INVOLUCRA LA REALIZACIÓN ...,FACULTAD DE ODONTOLOGÍA,DEPARTAMENTO DE ASTRONOMÍA,promoting reflection in preclinical dental stu...,the present project involves the realization o...,,faculty of dentistry,department of astronomy,True,title: promoting reflection in preclinical den...
3,218.163.016-INI,INDEFINIDO,,MOTIVACIÓN Y HABILIDADES SOCIALES EN ADOLESCENTES,,EL ESTUDIO DE LA MOTIVACIÓN TIENE DIFERENTES A...,FACULTAD DE EDUCACIÓN,"DEPARTAMENTO DE CIENCIAS DE LA EDUCACIÓN, DEPT...",motivation and social skills in adolescents,the study of the motivation has different side...,,faculty of education,"department of education sciences, department o...",True,title: motivation and social skills in adolesc...
4,219.091.052-INI,NO,,TIME EFFECTS ON THE LIQUEFACTION RESPONSE OF G...,,SECONDARY CONSOLIDATION AND AGEING ARE TWO OFT...,FACULTAD DE INGENIERÍA,DEPTO. TEORÍA POLITICA Y FUND.DE LA EDUC.,time effects on the liquefaction response of g...,secondary consolidation and ageing are two oft...,,faculty of engineering,department of political and fund theory of edu...,False,title: time effects on the liquefaction respon...


# 3) Split dataset

In [221]:
def to_serializable(obj):
    if hasattr(obj, "tolist"):
        return obj.tolist()
    return obj

def count_and_eval(labels): 
    c = Counter(labels)
    p = c[0]/c[1]
    print(c, p)
    

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
import json
from collections import Counter

path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

df = df[df["Interdisciplinario"] != "INDEFINIDO"]
le = LabelEncoder()

df["labels"] = le.fit_transform(df["Interdisciplinario"])

# Crear la clase combinada como categoría
df["stratify"] = df[["labels", "Español"]].astype(str).agg("_".join, axis=1)
# Codificar en números
df["stratify"] = df["stratify"].astype("category").cat.codes

# Convertir a arrays
ids = df["Código VRID"].to_numpy()
strat_classes = df["stratify"].to_numpy()
labels = df["labels"].to_numpy()

In [224]:
# Train/Test split (ids y labels en paralelo)
idx_train, idx_test, y_train, y_test = train_test_split(
    labels,
    strat_classes,
    test_size=0.2,
    random_state=7,
    stratify=strat_classes
)

print(ids.shape)
print(idx_train.shape)
print(idx_train.shape[0]+idx_test.shape[0])

print("Train:", Counter(y_train))
print("Test:", Counter(y_test))

print("Train:", Counter(idx_train))
print("Test:", Counter(idx_test))

count_and_eval(idx_train)
count_and_eval(idx_test)

"""
# Crear folds sobre train
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)

folds = []
for fold, (train_pos, val_pos) in enumerate(skf.split(idx_train, y_train)):
    train_ids = idx_train[train_pos]   # array de IDs
    val_ids = idx_train[val_pos]       # array de IDs
    folds.append(val_ids)

print("Test size:", len(idx_test))
print("Fold 0 - Val size:", len(folds[0]))

#Guardar index en diccionario
dataset_index = {
    "Train": idx_train,
    "Test": idx_test,
    "kfolds": folds 
}
filepath=os.path.join(path, "train_test_ids_3folds.json")

# Guardar
#with open(filepath, "w", encoding="utf-8") as f:
#    json.dump(dataset_index, f, default=to_serializable, indent=2, ensure_ascii=False)
"""


(964,)
(771,)
964
Train: Counter({3: 295, 0: 169, 1: 158, 2: 149})
Test: Counter({3: 74, 0: 43, 1: 39, 2: 37})
Train: Counter({1: 444, 0: 327})
Test: Counter({1: 111, 0: 82})
Counter({1: 444, 0: 327}) 0.7364864864864865
Counter({1: 111, 0: 82}) 0.7387387387387387


'\n# Crear folds sobre train\nskf = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)\n\nfolds = []\nfor fold, (train_pos, val_pos) in enumerate(skf.split(idx_train, y_train)):\n    train_ids = idx_train[train_pos]   # array de IDs\n    val_ids = idx_train[val_pos]       # array de IDs\n    folds.append(val_ids)\n\nprint("Test size:", len(idx_test))\nprint("Fold 0 - Val size:", len(folds[0]))\n\n#Guardar index en diccionario\ndataset_index = {\n    "Train": idx_train,\n    "Test": idx_test,\n    "kfolds": folds \n}\nfilepath=os.path.join(path, "train_test_ids_3folds.json")\n\n# Guardar\n#with open(filepath, "w", encoding="utf-8") as f:\n#    json.dump(dataset_index, f, default=to_serializable, indent=2, ensure_ascii=False)\n'

# 4) TF-ID feature extractor 

In [200]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
#filepath=os.path.join(path, "train_test_ids_3folds.json")
#with open(filepath, "r", encoding="utf-8") as f:
#    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

#Prueba con solo textos traducidos
#df = df[df["Español"]==False]

## Eval duplicados

In [146]:
from sklearn.preprocessing import LabelEncoder
from models.TIFD import gen_TFID_dataset, preprocess_text_for_TFID, gen_TFID_vectors
import numpy as np

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_TFID_dataset(codes_test, df)
#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_TFID_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]



In [147]:
eval_duplicate(X_train, X_test)

Duplicados en train: False
Duplicados en test: False
Hay intersección: False


## Original

In [201]:
from sklearn.preprocessing import LabelEncoder
from models.TIFD import gen_TFID_dataset, preprocess_text_for_TFID, gen_TFID_vectors
import numpy as np

df = df[df["Español"]==False]
#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_TFID_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_TFID_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

#Lemantización y eliminación de stopwords
X_train = preprocess_text_for_TFID(X_train)
# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)
#Creacion de vectores TFID
X_train, X_test = gen_TFID_vectors(X_train, X_test)

In [202]:
from utils.cv import CvCustom
from collections import Counter

cv_function=CvCustom(df_decode)
for train_idx, test_idx in cv_function.split(X_train, y_train):
    print(train_idx.shape, test_idx.shape)
    #eval_duplicate(train_idx, test_idx)
    print("train:", Counter(y_train[train_idx]))
    print("test:", Counter(y_train[test_idx]))

(254,)
(177,) (77,)
train: Counter({0: 94, 1: 83})
test: Counter({0: 41, 1: 36})
(171,) (83,)
train: Counter({0: 92, 1: 79})
test: Counter({0: 43, 1: 40})
(160,) (94,)
train: Counter({0: 84, 1: 76})
test: Counter({0: 51, 1: 43})


In [203]:
c = Counter(y_train)
print("train:", c)
print(c[0]/c[1])
c = Counter(y_test)
print("test:", c)
print(c[0]/c[1])

train: Counter({0: 169, 1: 149})
1.1342281879194631
test: Counter({0: 43, 1: 37})
1.162162162162162


## Train

In [68]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model, eval_model, mlflow_ckeckpoint
from utils.cv import CvCustom
from collections import Counter

# 2. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    #'DecisionTreeClassifier',
    'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=2
sample_weight_On=True
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
(771,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.64, 'std_test_score': 0.02}
RandomForestClassifier: {'mean_test_score': 0.62, 'std_test_score': 0.02}
XGBClassifier: {'mean_test_score': 0.61, 'std_test_score': 0.02}
SVC: {'mean_test_score': 0.62, 'std_test_score': 0.0}


In [69]:
# Métricas por idioma
lang_es = df_test["Español"]

for name, model in models_dicc.items():
    results=eval_model(model, X_test, y_test, lang_es)
    print(results)

{'accuracy': 0.6994818652849741, 'precision': 0.7572815533980582, 'recall': 0.7027027027027027, 'f1_score': 0.7008534977190135, 'cm': array([[57, 25],
       [33, 78]]), 'f1_es': 0.680792379812061, 'f1_en': 0.7282859686012461, 'cm_es': array([[23, 17],
       [20, 55]]), 'cm_en': array([[34,  8],
       [13, 23]])}
{'accuracy': 0.7046632124352331, 'precision': 0.6875, 'recall': 0.8918918918918919, 'f1_score': 0.6865743315085002, 'cm': array([[37, 45],
       [12, 99]]), 'f1_es': 0.6480027056480403, 'f1_en': 0.7053705787883002, 'cm_es': array([[ 9, 31],
       [ 3, 72]]), 'cm_en': array([[28, 14],
       [ 9, 27]])}
{'accuracy': 0.6217616580310881, 'precision': 0.6484375, 'recall': 0.7477477477477478, 'f1_score': 0.6133427247370429, 'cm': array([[37, 45],
       [28, 83]]), 'f1_es': 0.5978024059865492, 'f1_en': 0.6285107297765526, 'cm_es': array([[12, 28],
       [16, 59]]), 'cm_en': array([[25, 17],
       [12, 24]])}
{'accuracy': 0.6735751295336787, 'precision': 0.7553191489361702, 'r

In [96]:
import mlflow
import git 
#MLflow logging helper function
def safe_log_metric(name, value):
    try:
        if isinstance(value, (list, tuple, np.ndarray)):
            if np.size(value) == 1:
                value = float(np.array(value).item())
            else:
                raise ValueError("Métrica con más de un valor.")
        else:
            value = float(value)
        mlflow.log_metric(name, value)
    except Exception as e:
        print(f"⚠️ No se pudo loggear {name}: {e}")

def mlflow_ckeckpoint(exp_info, results_val, models_dicc, extra_parms, X_test, y_test, lang_es):
    # Set backend store
    #mlflow.set_tracking_uri(uri="http://localhost:5000")
    #mlflow.set_tracking_uri("http://172.17.0.1:5000")
    mlflow.set_tracking_uri("http://mlflow-server:5000")
    #mlflow.set_tracking_uri("http://172.17.0.1:5000")
    tracking_uri = mlflow.get_tracking_uri()
    print("Current tracking uri: {}".format(tracking_uri)) 

    # Define el experimento (lo crea si no existe)
    mlflow.set_experiment(exp_info["exp_name"])
    
    # Obtener commit actual
    repo = git.Repo(search_parent_directories=True)
    commit_hash = repo.head.object.hexsha

    for model_name, metrics in results_val.items():
        model = models_dicc[model_name]

        with mlflow.start_run(run_name=model_name):
            print(f"📝 Registrando modelo en MLflow: {model_name}")

            # Hiperparámetros
            try:
                mlflow.log_params(model.get_params())
            except:
                print(f"⚠️ No se pudieron loggear los hiperparámetros para {model_name}")

            #Parámetros adicionales
            for k, v in extra_parms.items():
                mlflow.log_param(k, v)

            # Métricas de validación
            for k, v in metrics.items():
                safe_log_metric(f"val_{k}", v)

            # Métricas de test
            results_test = eval_model(model, X_test, y_test, lang_es)
            for k, v in results_test.items():
                if k.startswith("cm"):
                    # Guardar confusion matrix (o similar) como artefacto
                    # Guardar como CSV temporal
                    fname = f"{k}.csv"
                    np.savetxt(fname, v, delimiter=",", fmt="%d")

                    mlflow.log_artifact(fname, artifact_path="confusion_matrices")

                    # Eliminar archivo local si no lo necesitas
                    os.remove(fname)

                else:
                    # Guardar métrica numérica
                    safe_log_metric(f"test_{k}", v)
            
            #Guardar commit de git
            mlflow.log_param("git_commit", commit_hash)
                    
            # Guardar modelo
            mlflow.sklearn.log_model(model, name = "model", input_example=X_test[:5])


In [99]:
exp_info = {
    'exp_name': "Bayesiansearchcv_TFID",
    #'artifact_path': "file:///tmp/mlflow_experiments/mlruns", 
    #'tracking_path': "sqlite:////tmp/mlflow_experiments/mlflow.db",
}

extra_parms = {
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, extra_parms, X_test, y_test, lang_es)

2025/08/22 23:28:44 INFO mlflow.tracking.fluent: Experiment with name 'Bayesiansearchcv_TFID' does not exist. Creating a new experiment.


Current tracking uri: http://mlflow-server:5000
📝 Registrando modelo en MLflow: LogisticRegression
🏃 View run LogisticRegression at: http://mlflow-server:5000/#/experiments/513370166166108560/runs/b84577b59c5f422eac8179289dda10d5
🧪 View experiment at: http://mlflow-server:5000/#/experiments/513370166166108560
📝 Registrando modelo en MLflow: RandomForestClassifier
🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/513370166166108560/runs/0fcf1d42720c4034ac23314acfeabc1f
🧪 View experiment at: http://mlflow-server:5000/#/experiments/513370166166108560
📝 Registrando modelo en MLflow: XGBClassifier
🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/513370166166108560/runs/4c9f9413d1bd48fcb98d6b1add1c4fb2
🧪 View experiment at: http://mlflow-server:5000/#/experiments/513370166166108560
📝 Registrando modelo en MLflow: SVC
🏃 View run SVC at: http://mlflow-server:5000/#/experiments/513370166166108560/runs/737697b9464e41a383a018a3ee5b35f9
🧪 View experi